In [7]:
import pandas as pd 
import pdfplumber 

In [8]:
path = "raw_data_2025/20250217_dane_szacunkowe_za_I_2025.pdf"


with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    table = page.find_tables()[0]
    # granice kolumn z krawędzi komórek pierwszego wiersza danych
    xs = sorted({round(c[0], 1) for c in table.cells} | {round(c[2], 1) for c in table.cells})
    print("Granice kolumn:", xs)
    x_lp, x_name, x_budget, x_exec, x_pct, x_end = xs[:6]
    top = table.bbox[1]
    words = [w for w in page.extract_words(keep_blank_chars=False, use_text_flow=False)
             if w["top"] > top + 60 and x_lp <= w["x0"] and w["bottom"] <= table.bbox[3]]

# grupowanie słów w wiersze po pozycji pionowej
words.sort(key=lambda w: (round(w["top"]), w["x0"]))
lines = []
for w in words:
    if lines and abs(lines[-1]["top"] - w["top"]) < 3:
        lines[-1]["words"].append(w)
    else:
        lines.append({"top": w["top"], "words": [w]})

rows = []
for ln in lines:
    cols = ["", "", "", "", ""]
    for w in sorted(ln["words"], key=lambda w: w["x0"]):
        c = w["x1"]
        i = 0 if c <= x_name else 1 if c <= x_budget and w["x0"] < x_budget - 40 else 2 if c <= x_exec else 3 if c <= x_pct else 4
        # słowa tekstowe zawsze do kolumny nazwy
        if i in (2, 3, 4) and w["x0"] < x_budget - 5 and not re.fullmatch(r"-?[\d,]+\)?|-", w["text"]):
            i = 1
        cols[i] = (cols[i] + " " + w["text"]).strip()
    rows.append(cols)

for r in rows:
    print(f"{r[0]:<6} | {r[1][:60]:<60} | {r[2]:>12} | {r[3]:>12} | {r[4]:>6}")

Granice kolumn: [56.2, 71.2, 323.4, 370.8, 422.5, 472.5]
1.     | DOCHODY                                                      |    632 848,2 |     47 390,9 |    7,5
       | z tego :                                                     |              |              |       
       | a) dochody podatkowe                                         |    570 772,2 |     43 175,6 |    7,6
       | w tym:                                                       |              |              |       
       | - podatki pośrednie                                          |    452 962,0 |     42 510,3 |    9,4
       | w tym: podatek akcyzowy                                      |     98 136,0 |      7 755,4 |    7,9
       | - podatek dochodowy od osób prawnych                         |     70 719,0 |    6 015,51) |    8,5
       | - podatek dochodowy od osób fizycznych                       |     31 735,6 |   -6 681,51) |      -
       | - podatek od niektórych instytucji finansowych               |

In [22]:
import re
from pathlib import Path

ROMAN = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6,
         "VII": 7, "VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12}
NUM_RE = re.compile(r"-?[\d ]*\d(?:,\d+)?|-")


def extract_raw_rows(path):
    """Zwraca listę [lp, nazwa, ustawa, wykonanie, proc] jako tekst, linia po linii."""
    with pdfplumber.open(path) as pdf:
        page = pdf.pages[0]
        table = page.find_tables()[0]
        xs = sorted({round(c[0], 1) for c in table.cells} | {round(c[2], 1) for c in table.cells})
        x_lp, x_name, x_budget, x_exec = xs[:4]
        x_pct = xs[-2]  # w części PDF-ów między wykonaniem a % jest wąska komórka z przypisem
        words = [w for w in page.extract_words()
                 if x_lp <= w["x0"] and w["bottom"] <= table.bbox[3]]
        # nagłówek ma różną wysokość w różnych miesiącach, tabela zaczyna się od "DOCHODY"
        start = next(w["top"] for w in words if w["text"] == "DOCHODY") - 3
        words = [w for w in words if w["top"] >= start]
        # samodzielne znaczniki przypisów ("*)", "1)") w kolumnach z liczbami
        words = [w for w in words
                 if not (w["x0"] >= x_budget and re.fullmatch(r"(\d|\*+)\)", w["text"]))]

    words.sort(key=lambda w: (round(w["top"]), w["x0"]))
    lines = []
    for w in words:
        if lines and abs(lines[-1][0] - w["top"]) < 3:
            lines[-1][1].append(w)
        else:
            lines.append((w["top"], [w]))

    rows = []
    for _, ws in lines:
        cols = [""] * 5
        for w in sorted(ws, key=lambda w: w["x0"]):
            is_num = re.fullmatch(r"-?[\d,]*\d[\d,]*\)?|-", w["text"])
            if w["x1"] <= x_name:
                i = 0
            elif w["x1"] <= x_budget or (not is_num and w["x0"] < x_budget - 5):
                i = 1
            else:
                i = 2 if w["x1"] <= x_exec else 3 if w["x1"] <= x_pct else 4
            cols[i] = (cols[i] + " " + w["text"]).strip()
        rows.append(cols)
    return rows


def to_float(s):
    s = re.sub(r"(,\d)\d\)$", r"\1", s.strip())  # przypis "1)" doklejony do liczby
    if s in ("", "-"):
        return float("nan")
    return float(s.replace(" ", "").replace(",", "."))


def clean_rows(rows):
    """Scala wieloliniowe etykiety i usuwa wiersze nagłówkowe bez wartości."""
    items = []  # [lp, nazwa, liczby] dla wierszy z wartościami
    gap = []  # linie samego tekstu między wierszami z wartościami
    pending_prefix = ""

    def flush_gap():
        nonlocal pending_prefix
        # przy >=2 liniach ostatnia jest początkiem etykiety następnego wiersza
        tail = gap[:-1] if len(gap) >= 2 else gap
        pending_prefix = gap[-1] if len(gap) >= 2 else ""
        if items and tail:
            items[-1][1] += " " + " ".join(tail)
        gap.clear()

    for lp, name, a, b, c in rows:
        if not (a or b or c):
            if not lp and not re.match(r"(z tego|w tym)\s*:?$", name):
                gap.append(name)
            continue
        flush_gap()
        if pending_prefix:
            name = pending_prefix + " " + name
            pending_prefix = ""
        items.append([lp, name, (a, b, c)])
    flush_gap()

    out = []
    for lp, name, nums in items:
        name = re.sub(r"\s(\d|\*+)\)$", "", name)
        name = re.sub(r"\s+w tym\s*:$", "", name).rstrip(" ,")
        m = re.match(r"(\d+\.\d+(?:\.\d+)?)\s+(.*)", name)
        if m:
            lp, name = m.group(1), m.group(2)
        # w części PDF-ów "4a." jest sklejone z tekstem ("4a.DEFICYTU")
        m = re.search(r"\b(\d+a)\.(?=\S)", name)
        if m:
            lp, name = m.group(1), name.replace(m.group(0), "")
        name = re.sub(r"^w tym:\s*", "", name).lstrip("- ").strip()
        out.append({"lp": lp.rstrip("."), "pozycja": name,
                    "ustawa": to_float(nums[0]), "wykonanie": to_float(nums[1]),
                    "proc_wykonania": to_float(nums[2])})
    return out


def load_month(path):
    rok_miesiac = re.search(r"za_([IVX]+)_(\d{4})", path.name)
    df = pd.DataFrame(clean_rows(extract_raw_rows(path)))
    df.insert(0, "nr", range(1, len(df) + 1))
    df.insert(0, "miesiac", ROMAN[rok_miesiac.group(1)])
    df.insert(0, "rok", int(rok_miesiac.group(2)))
    return df


df_2025 = pd.concat([load_month(p) for p in sorted(Path("raw_data_2025").glob("*.pdf"))],
                    ignore_index=True)
df_2025.groupby("miesiac").size()

miesiac
1     34
2     34
3     34
4     34
5     34
6     34
7     34
8     34
9     34
10    34
11    34
12    34
dtype: int64

In [21]:
out_dir = Path("data_2025")
out_dir.mkdir(exist_ok=True)
df_2025.to_csv(out_dir / "budzet_panstwa_2025.csv", index=False, encoding="utf-8")

# kontrola: te same pozycje w każdym miesiącu
print(df_2025.groupby("nr")["pozycja"].nunique().loc[lambda s: s > 1])
print(df_2025["wykonanie"].isna().groupby(df_2025["miesiac"]).sum().to_dict())
pd.read_csv(out_dir / "budzet_panstwa_2025.csv").query("miesiac == 1")

Series([], Name: pozycja, dtype: int64)
{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1}


,rok,miesiac,nr,lp,pozycja,ustawa,wykonanie,proc_wykonania
0,2025,1,1,1,DOCHODY,632848.2,47390.9,7.5
1,2025,1,2,NaN,a) dochody podatkowe,570772.2,43175.6,7.6
2,2025,1,3,NaN,podatki pośrednie,452962.0,42510.3,9.4
3,2025,1,4,NaN,podatek akcyzowy,98136.0,7755.4,7.9
4,2025,1,5,NaN,podatek dochodowy od osób prawnych,70719.0,6015.5,8.5
5,2025,1,6,NaN,podatek dochodowy od osób fizycznych,31735.6,-6681.5,NaN
6,2025,1,7,NaN,podatek od niektórych instytucji finansowych,6910.0,587.5,8.5
7,2025,1,8,NaN,b) dochody niepodatkowe,57940.6,4124.2,7.1
8,2025,1,9,NaN,wpływy z cła,7541.0,614.1,8.1
9,2025,1,10,NaN,c) środki z Unii Europejskiej i z innych źróde...,4135.4,91.0,2.2


In [25]:
df_2026 = pd.concat([load_month(p) for p in sorted(Path("raw_data_2026").glob("*.pdf"))],
                    ignore_index=True)

# W 2026 etykiety są łamane inaczej i parser je myli, ale tabela ma te same 34 wiersze
# co w 2025, więc etykiety bierzemy z grudnia 2025 (wartości liczbowe zostają z PDF-ów).
canon = df_2025.query("miesiac == 12").set_index("nr")[["lp", "pozycja"]]
lp_pdf = df_2026["lp"].replace("", pd.NA)
lp_canon = df_2026["nr"].map(canon["lp"])
assert (lp_pdf.isna() | (lp_pdf.astype(str) == lp_canon.astype(str))).all(), "numeracja wierszy się rozjeżdża"
df_2026["lp"] = lp_canon
df_2026["pozycja"] = df_2026["nr"].map(canon["pozycja"])

out_dir = Path("data_2026")
out_dir.mkdir(exist_ok=True)
df_2026.to_csv(out_dir / "budzet_panstwa_2026.csv", index=False, encoding="utf-8")

print(df_2026.groupby("miesiac").size().to_dict())
print(df_2026["wykonanie"].isna().groupby(df_2026["miesiac"]).sum().to_dict())
df_2026.query("miesiac == 9")

{1: 34, 2: 34, 3: 34, 4: 34, 5: 34, 6: 34, 7: 34, 8: 34, 9: 34}
{1: 3, 2: 2, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2}


,rok,miesiac,nr,lp,pozycja,ustawa,wykonanie,proc_wykonania
272,2026,9,1,1,DOCHODY,647200.0,438816.1,67.8
273,2026,9,2,,a) dochody podatkowe,579922.6,389180.3,67.1
274,2026,9,3,,podatki pośrednie,451253.0,317978.5,70.5
275,2026,9,4,,podatek akcyzowy,103253.0,68940.1,66.8
276,2026,9,5,,podatek dochodowy od osób prawnych,80423.1,62701.3,78.0
277,2026,9,6,,podatek dochodowy od osób fizycznych,32036.8,-7131.1,NaN
278,2026,9,7,,podatek od niektórych instytucji finansowych,7590.0,5647.0,74.4
279,2026,9,8,,b) dochody niepodatkowe,63643.1,47461.2,74.6
280,2026,9,9,,wpływy z cła,8733.0,6812.1,78.0
281,2026,9,10,,c) środki z Unii Europejskiej i z innych źróde...,3634.3,2174.6,59.8
